# Module 3: Cortex Code — Accelerating dbt Development

In this module we demonstrate how **Cortex Code** (CoCo) in Snowsight can dramatically accelerate data engineering workflows. We introduce two new data sources — **contract cancellations** and **customer NPS surveys** — and challenge Cortex Code to extend the existing `epower_dbt` pipeline with proper staging models, analytical marts, and tests.

---

### The Business Case

New data sources arrive constantly in any enterprise. A CRM team exports cancellation data. The marketing team starts collecting NPS surveys. The question every data team faces: **How fast can we onboard these new sources into our governed analytics pipeline?**

Traditionally this means: analyze the data, understand the schema, design staging models, write SQL transformations, configure dbt, add tests, validate results. Hours of work for a senior data engineer.

With Cortex Code, this becomes a **conversation**. CoCo reads the existing dbt project, understands the conventions, analyzes the new source tables, and generates production-ready dbt models — staging, marts, schema tests — all in minutes.

---

### What's New

| Source | Description | Rows | Key Metrics |
|--------|------------|------|-------------|
| **contract_cancellations** | Customer contract cancellations with reasons, retention attempts | ~2,400 | Churn rate, retention success, cancellation reasons |
| **customer_surveys** | NPS satisfaction surveys across 6 categories | 5,000 | NPS score, promoter/detractor ratio, category satisfaction |

Both join naturally with the existing `customer_dim` and `product_dim` — enabling cross-domain analytics like *"Do customers with low NPS scores have higher churn rates?"*

---

### Architecture

```
 EPOWER_BRONZE (Raw)              EPOWER_SILVER (Enriched)          EPOWER_GOLD (Business-Ready)
 ═══════════════════              ════════════════════════          ══════════════════════════════

 contract_cancellations ──►  stg_cancellations ──┬───────────►  mart_customer_churn
                              (+ customer_dim,   │               (by product, region, reason)
                               product_dim)      │
                                                 ├───────────►  mart_customer_health
                                                 │               (combined churn + NPS score)
 customer_surveys ────────►  stg_surveys ────────┼───────────►  mart_nps_analysis
                              (+ customer_dim,   │               (by region, type, category)
                               NPS segment)      │
                                                 └───────────►  mart_customer_health
```

---

### Snowflake Features Demonstrated

| Feature | Role in This Module |
|---------|-------------------|
| **Cortex Code (CoCo)** | AI-assisted dbt model generation — the star of this module |
| **dbt on Snowflake** | Native dbt project execution via `EXECUTE DBT PROJECT` |
| **Snowsight Workspace** | Integrated IDE where CoCo operates on the dbt project files |
| **Semantic Views** | (Bonus) Extend the agent with customer health analytics |

---

| Section | What we do |
|---------|------------|
| **§1** Prerequisites | Verify Module 1 is deployed |
| **§2** New Source Tables | Create tables for cancellations + surveys |
| **§3** Upload & Ingest | CSV → Stage → COPY INTO → verify |
| **§4** Explore the Data | Quick queries to understand the new sources |
| **§5** The CoCo Challenge | Guided instructions for Cortex Code dbt generation |
| **§6** Verification | Validate the dbt models CoCo created |
| **§7** Bonus: Semantic View + Agent | Extend the Intelligence Agent |

**Runtime**: ~15 minutes | **Prerequisite**: Module 1 (`epower_hol.ipynb`) must be deployed

## 1. Prerequisites

This module requires Module 1 (`epower_hol.ipynb`) to be fully deployed — we need the customer and product dimensions, the existing dbt project, and the EPOWER infrastructure.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;
USE DATABASE EPOWER_DEMO;

SELECT 'CUSTOMER_DIM' AS required_object, COUNT(*) AS "ROWS" FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
UNION ALL SELECT 'PRODUCT_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
UNION ALL SELECT 'SALES_FACT', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.SALES_FACT
UNION ALL SELECT 'REGION_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.REGION_DIM
ORDER BY required_object;

## 2. New Source Tables

We create two new tables in `EPOWER_BRONZE` — the raw ingestion layer. This follows the medallion architecture: raw data lands in Bronze, dbt transforms it through Silver (enriched) to Gold (business-ready).

| Table | Description | Key Columns |
|-------|------------|-------------|
| **CONTRACT_CANCELLATIONS** | Customer contract cancellations with German-language reasons | `customer_key`, `product_key`, `reason`, `retention_offered/accepted` |
| **CUSTOMER_SURVEYS** | NPS satisfaction surveys across 6 categories | `customer_key`, `nps_score` (0-10), `category`, `comment` (German) |

Both reference existing dimensions (`customer_dim`, `product_dim`) via foreign keys — enabling rich joins in the dbt models that Cortex Code will generate.

In [ ]:
%%sql
USE SCHEMA EPOWER_DEMO.EPOWER_BRONZE;

CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_BRONZE.CONTRACT_CANCELLATIONS (
    cancellation_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    cancellation_date DATE NOT NULL,
    reason VARCHAR(100) NOT NULL,
    channel VARCHAR(50) NOT NULL,
    retention_offered BOOLEAN NOT NULL,
    retention_accepted BOOLEAN NOT NULL
);

CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_BRONZE.CUSTOMER_SURVEYS (
    survey_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    survey_date DATE NOT NULL,
    nps_score INT NOT NULL,
    category VARCHAR(100) NOT NULL,
    comment VARCHAR(500)
);

## 3. Upload & Ingest

We upload the two new CSV files from the workspace to the existing internal stage (`@EPOWER_STAGE`), then load them into the tables we just created.

In [ ]:
import os

stage_name = '@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE'
base_path = '../demo_data/structured_data'

for csv_file in ['contract_cancellations.csv', 'customer_surveys.csv']:
    file_path = os.path.join(base_path, csv_file)
    stage_path = f'{stage_name}/structured_data/'
    session.file.put(file_path, stage_path, auto_compress=False, overwrite=True)
    print(f'  \u2713 {csv_file}')

print('\n\u2713 Files uploaded to stage')

In [ ]:
tables = ['contract_cancellations', 'customer_surveys']

print("Loading new source tables...")
for t in tables:
    try:
        r = session.sql(f"COPY INTO EPOWER_DEMO.EPOWER_BRONZE.{t} FROM @EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/structured_data/{t}.csv FILE_FORMAT=EPOWER_DEMO.EPOWER_OPS.CSV_FORMAT ON_ERROR='CONTINUE'").collect()
        print(f"  \u2713 {t}: {r[0]['rows_loaded']} rows")
    except Exception as e:
        print(f"  \u26a0 {t}: {e}")

print("\n\u2713 Data loading complete")

In [ ]:
%%sql
SELECT 'CONTRACT_CANCELLATIONS' AS table_name, COUNT(*) AS row_count FROM EPOWER_DEMO.EPOWER_BRONZE.CONTRACT_CANCELLATIONS
UNION ALL SELECT 'CUSTOMER_SURVEYS', COUNT(*) FROM EPOWER_DEMO.EPOWER_BRONZE.CUSTOMER_SURVEYS
ORDER BY table_name;

## 4. Explore the Data

Before we ask Cortex Code to build dbt models, let's understand what we loaded. These exploration queries also help frame the analytics requirements.

### Cancellation Reasons Breakdown

Why are customers leaving? This query shows the distribution of cancellation reasons (German-language: Umzug, Preiserh&#246;hung, Wettbewerber, etc.) along with how often retention was offered and accepted. Umzug (relocation) is typically the #1 reason in German energy retail — these customers aren't dissatisfied, they simply moved.

In [ ]:
%%sql
SELECT
    reason,
    COUNT(*) AS cancellations,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
    SUM(CASE WHEN retention_offered THEN 1 ELSE 0 END) AS retention_offered,
    SUM(CASE WHEN retention_accepted THEN 1 ELSE 0 END) AS retention_accepted
FROM EPOWER_DEMO.EPOWER_BRONZE.CONTRACT_CANCELLATIONS
GROUP BY reason
ORDER BY cancellations DESC;

### Cancellations by Product Category

Which product categories have the highest churn? This query joins cancellations with the product dimension to show cancellation counts per category. Expect electricity tariffs to dominate (easy to switch providers in Germany's liberalized market), while hardware products like solar and heat pumps have very low churn due to high switching costs.

In [ ]:
%%sql
SELECT
    p.category_name,
    COUNT(cc.cancellation_id) AS cancellations,
    COUNT(DISTINCT cc.customer_key) AS unique_customers_churned
FROM EPOWER_DEMO.EPOWER_BRONZE.CONTRACT_CANCELLATIONS cc
JOIN EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM p ON cc.product_key = p.product_key
GROUP BY p.category_name
ORDER BY cancellations DESC;

### NPS Segment Distribution

How satisfied are EPOWER's customers overall? This query classifies all survey responses into the standard NPS segments — **Promoters** (score 9-10), **Passives** (7-8), and **Detractors** (0-6) — and shows the count and average score per segment.

In [ ]:
%%sql
SELECT
    CASE
        WHEN nps_score >= 9 THEN 'Promoter (9-10)'
        WHEN nps_score >= 7 THEN 'Passive (7-8)'
        ELSE 'Detractor (0-6)'
    END AS nps_segment,
    COUNT(*) AS responses,
    ROUND(AVG(nps_score), 1) AS avg_score
FROM EPOWER_DEMO.EPOWER_BRONZE.CUSTOMER_SURVEYS
GROUP BY nps_segment
ORDER BY avg_score DESC;

### NPS by Survey Category

Which areas of the business are customers most and least satisfied with? This query breaks down NPS scores across the 6 survey categories (Gesamtzufriedenheit, Kundenservice, Produkt, Installation, Preis-Leistung, App & Digital) — showing average NPS and the promoter/detractor split for each.

In [ ]:
%%sql
SELECT
    category,
    COUNT(*) AS responses,
    ROUND(AVG(nps_score), 1) AS avg_nps,
    SUM(CASE WHEN nps_score >= 9 THEN 1 ELSE 0 END) AS promoters,
    SUM(CASE WHEN nps_score <= 6 THEN 1 ELSE 0 END) AS detractors
FROM EPOWER_DEMO.EPOWER_BRONZE.CUSTOMER_SURVEYS
GROUP BY category
ORDER BY avg_nps DESC;

## 5. The CoCo Challenge — Extend the dbt Pipeline

This is the core of this module. You'll use **Cortex Code** in the Snowsight workspace to extend the existing `epower_dbt` project with the two new data sources we just loaded.

---

### Step 1: Open Cortex Code

1. Navigate to **Projects → Workspaces** in Snowsight
2. Open the **EPOWER_Demo** workspace
3. Navigate to the `epower_dbt/` folder
4. Open the **Cortex Code** panel (chat icon in the sidebar)

---

### Step 2: Prompt CoCo

Copy and paste the following prompt into Cortex Code:

```
We just loaded two new tables into EPOWER_DEMO.EPOWER_BRONZE:

- CONTRACT_CANCELLATIONS — tracks when and why customers cancel contracts
- CUSTOMER_SURVEYS — NPS satisfaction surveys with scores and comments

Explore these tables and the existing dbt project, then extend the pipeline
with appropriate staging models and analytical marts. We want to understand
churn patterns and customer satisfaction across our business.
```

> **Why this prompt?** A realistic user knows the table names and business purpose, but lets CoCo discover the schema, figure out join keys, decide on model structure, and follow existing conventions — all on its own.

---

### Step 3: Review & Apply

Cortex Code will explore the source tables and propose a set of changes. Typically you should see:

- **Source definitions** — the new tables added to `sources.yml`
- **Staging models** — enriched versions joining with existing dimensions (e.g., customer context, product names)
- **Mart models** — aggregated business metrics (churn analysis, NPS breakdown, potentially a combined view)
- **Schema tests** — data quality assertions (`not_null`, `unique`, `accepted_values`)
- **Config updates** — `dbt_project.yml` updated for the new model paths

Review the generated code, then **apply all changes** in the workspace.

> **Presenter tip:** The exact model names and structure may vary — that's the point. CoCo reasons about the data and proposes its own design. Check the `reference/` folder for one expected solution.

---

### Step 4: Run & Test

In the Snowsight workspace dbt UI:

1. Click **Run** to materialize all models (including the new ones)
2. Click **Test** to validate data quality assertions
3. Check the DAG visualization — the new `customer_analytics` branch should appear alongside the existing `epulse_vpp` and `energy_market_data` branches

---

### What to Watch For

During the demo, highlight these points:

- **Schema discovery**: CoCo explores the tables, discovers columns and data types, and understands their business meaning
- **Convention adherence**: CoCo reads the existing project and follows naming patterns, schema targets, and materialization strategies
- **Join intelligence**: CoCo discovers join keys (`customer_key`, `product_key`) by analyzing the tables and links to existing dimensions
- **Test generation**: CoCo adds appropriate `not_null`, `unique`, and `accepted_values` tests
- **Cross-domain thinking**: CoCo may combine churn + NPS into a single model — it understands relationships across the new sources
- **Speed**: What would take a data engineer hours happens in minutes

---

> **Presenter tip:** If CoCo's output needs small adjustments, that's actually a good demo moment — show that CoCo gets you 90% there, and the last 10% is human review. This is the realistic AI-assisted workflow.

### Alternative: Deploy via SQL

After CoCo has generated the models and you've run them in the workspace UI, you can also re-deploy the dbt project as a Snowflake object (updating the existing deployment from Module 1):

In [ ]:
%%sql
CREATE OR REPLACE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT
    FROM 'snow://workspace/USER$.PUBLIC."EPOWER_Demo"/versions/live/epower_dbt';

EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'run';

EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'test';

## 6. Verification

After Cortex Code has generated the models and you've executed the dbt build, let's verify what was created.

In [ ]:
%%sql
-- Discover new staging models in SILVER
SELECT TABLE_NAME, ROW_COUNT, CREATED
FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'EPOWER_SILVER'
  AND TABLE_NAME ILIKE ANY ('%CANCEL%', '%SURVEY%', '%CHURN%', '%NPS%')
ORDER BY CREATED DESC;

In [ ]:
%%sql
-- Discover new mart models in GOLD
SELECT TABLE_NAME, ROW_COUNT, CREATED
FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'EPOWER_GOLD'
  AND TABLE_NAME ILIKE ANY ('%CHURN%', '%NPS%', '%HEALTH%', '%CANCEL%', '%SURVEY%')
ORDER BY CREATED DESC;

In [ ]:
%%sql
-- Sample the churn/cancellation mart (adjust table name if CoCo chose differently)
SELECT * FROM EPOWER_DEMO.EPOWER_GOLD.MART_CUSTOMER_CHURN
ORDER BY cancellations DESC
LIMIT 10;

In [ ]:
%%sql
-- Sample the NPS mart (adjust table name if CoCo chose differently)
SELECT * FROM EPOWER_DEMO.EPOWER_GOLD.MART_NPS_ANALYSIS
LIMIT 10;

In [ ]:
%%sql
-- Sample the combined health mart (adjust table name if CoCo chose differently)
SELECT * FROM EPOWER_DEMO.EPOWER_GOLD.MART_CUSTOMER_HEALTH
LIMIT 10;

## 7. Bonus: Extend the AI Layer with CoCo

Now that the new dbt models are materialized, let's use Cortex Code again — this time to make the data queryable in natural language.

---

### Step 1: Create a Semantic View

In Cortex Code, prompt:

```
I've just built new mart tables for customer churn and NPS analysis
in EPOWER_GOLD (check what tables exist there with ILIKE '%CHURN%'
or '%NPS%' or '%HEALTH%').
Create a Semantic View called CUSTOMER_HEALTH_SEMANTIC_VIEW in
EPOWER_DEMO.EPOWER_GOLD that covers these marts — include relevant
facts, dimensions, and metrics with German synonyms (this is a
German energy company). Look at the existing semantic views in
EPOWER_GOLD for style reference.
```

Review and execute the generated SQL.

---

### Step 2: Update the Agent

Then prompt CoCo:

```
Add the new CUSTOMER_HEALTH_SEMANTIC_VIEW as a tool called
customer_health_analyst to the existing EPOWER_AGENT in
EPOWER_DEMO.EPOWER_GOLD. Preserve all existing tools. You can
inspect the current agent definition with DESCRIBE AGENT.
```

Review and execute.

> **Presenter tip:** This demonstrates the full CoCo workflow — from raw data to dbt pipeline to semantic layer to agent — all driven by natural language prompts.

### Fallback: Manual SQL

> If CoCo's output needs adjustment, or you prefer to run pre-built SQL, the cells below contain reference implementations. These match the expected output from the `reference/` folder.

**Semantic View:**

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_HEALTH_SEMANTIC_VIEW
  TABLES (
    CHURN AS EPOWER_DEMO.EPOWER_GOLD.MART_CUSTOMER_CHURN
      PRIMARY KEY (CATEGORY_NAME, REGION_NAME, CANCELLATION_MONTH, REASON)
      WITH SYNONYMS = ('Kuendigung', 'cancellation', 'churn', 'Abwanderung'),
    NPS AS EPOWER_DEMO.EPOWER_GOLD.MART_NPS_ANALYSIS
      PRIMARY KEY (REGION_NAME, CUSTOMER_TYPE, CATEGORY, SURVEY_MONTH)
      WITH SYNONYMS = ('Zufriedenheit', 'satisfaction', 'NPS', 'survey')
  )
  FACTS (
    CHURN.CANCELLATIONS AS CANCELLATIONS
      WITH SYNONYMS = ('Kuendigungen', 'churn count')
      COMMENT = 'Number of contract cancellations',
    CHURN.RETENTION_OFFERED_COUNT AS RETENTION_OFFERED_COUNT
      COMMENT = 'Number of retention offers made',
    CHURN.RETENTION_ACCEPTED_COUNT AS RETENTION_ACCEPTED_COUNT
      COMMENT = 'Number of retention offers accepted',
    NPS.TOTAL_RESPONSES AS TOTAL_RESPONSES
      COMMENT = 'Total NPS survey responses',
    NPS.AVG_NPS AS AVG_NPS
      WITH SYNONYMS = ('average NPS', 'Durchschnitt NPS')
      COMMENT = 'Average NPS score (0-10)',
    NPS.PROMOTERS AS PROMOTERS
      COMMENT = 'Count of promoters (NPS 9-10)',
    NPS.DETRACTORS AS DETRACTORS
      COMMENT = 'Count of detractors (NPS 0-6)'
  )
  DIMENSIONS (
    CHURN.CATEGORY_NAME AS CHURN_CATEGORY
      WITH SYNONYMS = ('Produktkategorie', 'product category')
      COMMENT = 'Product category of cancelled contract',
    CHURN.REGION_NAME AS CHURN_REGION
      WITH SYNONYMS = ('Region', 'Gebiet')
      COMMENT = 'Region where cancellation occurred',
    CHURN.CANCELLATION_MONTH AS CANCELLATION_MONTH
      WITH SYNONYMS = ('Monat', 'month')
      COMMENT = 'Month of cancellation',
    CHURN.REASON AS CANCELLATION_REASON
      WITH SYNONYMS = ('Kuendigungsgrund', 'reason', 'Grund')
      COMMENT = 'Reason for cancellation (Umzug, Preiserhoehung, Wettbewerber, ...)',
    NPS.REGION_NAME AS NPS_REGION
      COMMENT = 'Region of survey respondent',
    NPS.CUSTOMER_TYPE AS NPS_CUSTOMER_TYPE
      WITH SYNONYMS = ('Kundentyp', 'segment')
      COMMENT = 'Customer type (Privatkunde, Kleingewerbe, Gewerbekunde)',
    NPS.CATEGORY AS NPS_CATEGORY
      WITH SYNONYMS = ('Umfragekategorie', 'survey topic')
      COMMENT = 'Survey category (Gesamtzufriedenheit, Kundenservice, ...)',
    NPS.SURVEY_MONTH AS SURVEY_MONTH
      COMMENT = 'Month of survey response'
  )
  METRICS (
    CHURN.TOTAL_CANCELLATIONS AS SUM(CHURN.CANCELLATIONS)
      WITH SYNONYMS = ('Gesamtkuendigungen', 'total churn')
      COMMENT = 'Total contract cancellations',
    NPS.OVERALL_AVG_NPS AS AVG(NPS.AVG_NPS)
      WITH SYNONYMS = ('Gesamt-NPS', 'overall satisfaction')
      COMMENT = 'Overall average NPS score'
  )
  COMMENT = 'Customer health analytics — churn analysis and NPS satisfaction scores';

**Agent Definition:**

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, customer portal engagement, customer health (churn + NPS), and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions -> energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products -> customer_energy_analyst
    - Sales/contracts -> energy_sales_analyst
    - Billing -> billing_analyst
    - Service tickets -> service_analyst
    - HR data -> hr_analyst
    - Electricity market prices, day-ahead -> market_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export -> vpp_telemetry_analyst
    - Portal activity, digital engagement -> portal_analyst
    - Customer churn, cancellations, NPS, satisfaction -> customer_health_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: portal_analyst, description: "Customer portal engagement: logins, meter readings, tariff changes, service requests."}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_health_analyst, description: "Customer churn analysis, contract cancellations, NPS surveys, customer satisfaction, retention metrics."}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  portal_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW"}
  customer_health_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_HEALTH_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

### Demo Questions for the Agent

| # | Question | What it tests |
|---|----------|---------------|
| 1 | *"What are the top cancellation reasons?"* | Basic churn metric |
| 2 | *"Welche Produktkategorie hat die höchste Kündigungsrate?"* | Category churn (German) |
| 3 | *"What is our overall NPS score?"* | NPS aggregation |
| 4 | *"Compare customer satisfaction between regions"* | Regional NPS comparison |
| 5 | *"How effective are our retention offers?"* | Retention analysis |
| 6 | *"Show me customers with high churn risk and low NPS"* | **Cross-domain**: churn + NPS combined |

&nbsp;

> **Presenter tip:** Question 6 combines cancellation data with NPS surveys — demonstrating that CoCo's `mart_customer_health` model enables cross-domain analytics that weren't possible before.

---

## Summary

In this module you demonstrated the **Cortex Code end-to-end workflow**:

| What | How |
|------|-----|
| **New data sources** | Contract cancellations + NPS surveys loaded from CSV |
| **AI-assisted engineering** | Cortex Code generated dbt models, semantic views, and agent config |
| **Schema discovery** | CoCo explored tables, discovered joins, and proposed model structure |
| **Convention adherence** | CoCo followed existing project naming, materialization, and schema patterns |
| **Cross-domain analytics** | Combined churn + NPS into customer health analytics |
| **AI-ready** | Semantic View + Cortex Agent — queryable in natural language |

**The key insight:** With Cortex Code, onboarding new data sources — from raw tables through a governed dbt pipeline to a natural-language-queryable agent — takes minutes, not hours. The data engineer states the business goal; CoCo figures out the implementation.

---

*EPOWER Module 3 — Cortex Code + dbt — Powered by Snowflake*